**Autores**

* DIANA CAROLINA LEAL MARTINEZ
* DIEGO ALEJANDRO FIGUEREDO SALAMANCA
* MARIA CAMILA REYES GIL
* SILVIA CARDENAS SANTOS

# ETL de TiposBeneficio, MiniCondicionesTipoBeneficio y Proveedor

Este notebook implementa la extracción, transformación y carga de tres tablas del modelo multidimensional de RaSA:

| Tabla destino | Fuente | Columnas | Papel en el modelo |
| :--- | :--- | :--- | :--- |
| `TiposBeneficio_ELT` | F2 `FuenteTiposBeneficio_ETL` | `IdTipoBeneficio_DWH`, `IdTipoBeneficio_T`, `Nombre` | Dimensión de los tipos de beneficio |
| `MiniCondicionesTipoBeneficio_ELT` | F2 `FuenteTiposBeneficio_ETL` | `IdCondicionesBeneficios_DWH`, las cinco condiciones Yes/No y `UnidadDelLimite` | Minidimensión (historia tipo 4) de las condiciones que cambian de un año a otro |
| `Proveedor_ELT` | F3 `FuentePlanesBeneficio_ETL` | `IdProveedor_DWH`, `IdProveedor_T` | Dimensión de proveedores, solo con identificadores (nota 2 del modelo) |

**Proceso de implementación**

1. Extraer F2 completa y de F3 solo las columnas que se necesitan (`IdProveedor_T`, `IdPlan_T`, `IdTipoBeneficio_T` y `Fecha`).
2. Transformar aplicando las reglas de negocio de RaSA:
   * eliminar duplicados y espacios sobrantes.
   * homologar las condiciones para que solo queden `Yes` y `No`.
   * reemplazar la unidad del límite vacía por `No aplica`.
   * unificar las unidades del límite que solo difieren en mayúsculas y minúsculas.
   * validar que exista una sola combinación de condiciones por tipo de beneficio y año.
   * completar el proveedor faltante con los 5 primeros caracteres de `IdPlan_T`.
3. Asignar las llaves sustitutas (`_DWH`) con `row_number()` ordenado por la llave natural. Así son consecutivas desde 1 y cada ejecución produce las mismas llaves.
4. Cargar en la base personal con el sufijo `_ELT`, reemplazando el contenido anterior, y verificar cuántas filas quedaron en cada tabla.

Manejo de historia (tipo 4). Las condiciones de un tipo de beneficio pueden cambiar de un año a otro, por eso no se guardan en `TiposBeneficio` sino en la minidimensión `MiniCondicionesTipoBeneficio`, que tiene **una fila por cada combinación distinta de condiciones** y no una por beneficio. La factless `HechoHistCondicionesTiposBeneficio` registra qué combinación tuvo cada beneficio en cada periodo y se carga junto con las tablas de hechos, para esas cargas este notebook construye el mapa `(IdTipoBeneficio_T, Anio) → IdCondicionesTipoBeneficio_DWH` y lo valida contra F3 (sección 3.7).

## 1. Conexión a la base de datos
Se configuran las conexiones JDBC a la base transaccional de origen y a la base personal de destino. Complete `db_user`, `db_psswd` y `db_destino` antes de ejecutar.

In [ ]:
# Configuración servidor base de datos transaccional
db_user = 'DB_202615'
db_psswd = ''
db_host = '157.253.236.120'
db_port = 8080
db_destino = 'DB_202615'      

if not db_user or not db_psswd or '<' in db_destino:
    raise ValueError('Complete db_user, db_psswd y db_destino antes de continuar')

jdbc_opciones = 'useConfigs=maxPerformance&connectTimeout=10000&socketTimeout=600000'
source_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/WWImportersTransactional?{jdbc_opciones}'
dest_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/{db_destino}?rewriteBatchedStatements=true&{jdbc_opciones}'

In [2]:
import os
import sys
from pathlib import Path
import urllib.request

# En la máquina virtual ya está todo instalado
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_CODESPACES = os.environ.get('CODESPACES') == 'true'

if IN_COLAB or IN_CODESPACES:
    %pip install -q pyspark==4.2.0 pandas==2.3.3 numpy==2.5.3 pyarrow==25.0.1

if IN_COLAB:
    # Spark 4 requiere Java 17 o superior
    !apt-get install -y -qq openjdk-21-jdk-headless > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'

ENVIRONMENT = 'colab' if IN_COLAB else 'codespaces' if IN_CODESPACES else 'local'
print(f'Entorno detectado: {ENVIRONMENT}')

from pyspark.sql import SparkSession, Window, functions as sf

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Necesario en Windows si Python está en C:\Program Files
import pyspark
os.environ.setdefault('SPARK_HOME', os.path.dirname(pyspark.__file__))

# Spark 4.2 en Windows requiere HADOOP_HOME
if os.name == 'nt' and 'HADOOP_HOME' not in os.environ:
    hadoop_home = Path.home() / '.miid4103' / 'hadoop'
    (hadoop_home / 'bin').mkdir(parents=True, exist_ok=True)
    try:
        (hadoop_home / 'bin' / 'winutils.exe').touch(exist_ok=True)
    except OSError:
        pass
    os.environ['HADOOP_HOME'] = str(hadoop_home)

CONNECTOR_VERSION = '9.7.0'
CONNECTOR_URL = ('https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/'
                 f'{CONNECTOR_VERSION}/mysql-connector-j-{CONNECTOR_VERSION}.jar')

def obtener_jar_mysql():
    env_path = os.environ.get('CONNECTOR_J_PATH', '')
    if env_path and Path(env_path).is_file():
        return Path(env_path)
    jar = Path.home() / '.miid4103' / 'jars' / f'mysql-connector-j-{CONNECTOR_VERSION}.jar'
    if not jar.exists():
        jar.parent.mkdir(parents=True, exist_ok=True)
        print(f'Descargando MySQL Connector/J {CONNECTOR_VERSION}...')
        parcial = jar.with_suffix('.part')
        urllib.request.urlretrieve(CONNECTOR_URL, parcial)
        parcial.replace(jar)
    return jar

MYSQL_JAR = obtener_jar_mysql()
MYSQL_JAR_URI = MYSQL_JAR.as_uri()

spark = (SparkSession.builder
         .master('local[*]')
         .appName('MIID4103-ETL-TiposBeneficio-Proveedor-SilviaCardenas')
         .config('spark.driver.host', '127.0.0.1')
         .config('spark.jars', MYSQL_JAR_URI)
         .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.ui.enabled', 'false')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.ansi.enabled', 'false')    # comportamiento de Spark 3.5
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('¡Spark inicializado correctamente con el conector JDBC!')

Entorno detectado: local


¡Spark inicializado correctamente con el conector JDBC!


### Funciones
`extraer_anio` toma el año de cuatro dígitos de una fecha, sin importar el formato en que venga (`2018`, `2018-12-31` o `Dec 31,2018`). `guardar_db` recibe el modo de escritura, con `overwrite` vacía la tabla y la vuelve a llenar sin cambiar su estructura.

In [3]:
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    df_bd = spark.read.format('jdbc')\
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()
    return df_bd.cache()

def guardar_db(db_connection_string, df, tabla, db_user, db_psswd, modo='append'):
    # Con modo='overwrite', truncate vacía la tabla y la vuelve a llenar sin cambiar su estructura
    df.select('*').write.format('jdbc') \
      .mode(modo) \
      .option('truncate', 'true') \
      .option('url', db_connection_string) \
      .option('dbtable', tabla) \
      .option('user', db_user) \
      .option('password', db_psswd) \
      .option('driver', 'com.mysql.cj.jdbc.Driver') \
      .option('batchsize', 10000) \
      .option('numPartitions', 4) \
      .save()

def extraer_anio(columna):
    return sf.regexp_extract(sf.col(columna).cast('string'), r'(\d{4})', 1).cast('int')

## 2. Extracción

### 2.1 Fuente de tipos de beneficio (F2)
Se extraen todas las columnas de F2. Los duplicados se eliminan en la transformación para poder reportar cuántos había.

In [4]:
sql_fuente_tipos_beneficio = '''
(SELECT
    IdTipoBeneficio_T,
    Nombre,
    UnidadDelLimite,
    EsEHB,
    EstaCubiertaPorSeguro,
    TieneLimiteCuantitativo,
    ExcluidoDelDesembolsoMaximoDentroDeLaRed,
    ExcluidoDelDesembolsoMaximoFueraDeLaRed,
    Fecha
FROM
    RaSaTransaccional_ETL.FuenteTiposBeneficio_ETL) AS tipos_beneficio
'''
df_tipos_beneficio_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_fuente_tipos_beneficio, db_user, db_psswd)
print(f'Registros extraídos de F2: {df_tipos_beneficio_raw.count():,}')
df_tipos_beneficio_raw.show(10)

Registros extraídos de F2: 449
+-----------------+--------------------+-------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|IdTipoBeneficio_T|              Nombre|    UnidadDelLimite|EsEHB|EstaCubiertaPorSeguro|TieneLimiteCuantitativo|ExcluidoDelDesembolsoMaximoDentroDeLaRed|ExcluidoDelDesembolsoMaximoFueraDeLaRed|Fecha|
+-----------------+--------------------+-------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----+
|                5|Abortion For Whic...|                   |   No|                   No|                     No|                                      No|                                     No| 2017|
|                5|Abortion For Whic...|                   |   No|                  Yes|                     No|                                      No|                

### 2.2 Fuente de planes de beneficio (F3)
De F3 solo se necesitan el proveedor y el plan, para construir `Proveedor` y el tipo de beneficio y la fecha, para validar el mapa de condiciones que usarán las tablas de hechos.

In [5]:
sql_fuente_planes_beneficio = '''
(SELECT
    IdProveedor_T,
    IdPlan_T,
    IdTipoBeneficio_T,
    Fecha
FROM
    RaSaTransaccional_ETL.FuentePlanesBeneficio_ETL) AS planes_beneficio
'''
df_planes_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_fuente_planes_beneficio, db_user, db_psswd)
print(f'Registros extraídos de F3: {df_planes_raw.count():,}')
df_planes_raw.show(10)

Registros extraídos de F3: 70,752
+-------------+-----------------+-----------------+----------+
|IdProveedor_T|         IdPlan_T|IdTipoBeneficio_T|     Fecha|
+-------------+-----------------+-----------------+----------+
|        16842|16842FL0070128-03|              640|2017-12-31|
|        14002|14002TN0400104-06|              335|2019-12-31|
|        19722|19722NM0010001-02|              885|2020-12-31|
|        81413|81413WI0460015-01|               10|2019-12-31|
|        52697|52697WI0010002-00|              190|2017-12-31|
|        28162|28162OH0060061-04|              810|2019-12-31|
|        20129|20129IL0330060-05|               10|2019-12-31|
|        40572|40572FL0070006-02|              440|2021-12-31|
|        38166|38166WI0180012-06|              620|2017-12-31|
|        70893|70893GA0010009-04|              815|2017-12-31|
+-------------+-----------------+-----------------+----------+
only showing top 10 rows


## 3. Transformación
### 3.1 Limpieza de F2
**Regla de negocio:** Puede que existan todavía duplicados de los registros en las fuentes, por favor revisar y eliminarlos.

Primero se quitan los espacios sobrantes de los textos, para que dos filas que solo difieren en espacios cuenten como duplicadas. El año de `Fecha` se guarda como entero en `Anio`.

In [6]:
CONDICIONES_YES_NO = ['EsEHB', 'EstaCubiertaPorSeguro', 'TieneLimiteCuantitativo',
                      'ExcluidoDelDesembolsoMaximoDentroDeLaRed', 'ExcluidoDelDesembolsoMaximoFueraDeLaRed']
ATRIBUTOS_CONDICION = CONDICIONES_YES_NO + ['UnidadDelLimite']

df_tb = df_tipos_beneficio_raw.withColumn('Nombre', sf.regexp_replace(sf.trim(sf.col('Nombre')), r'\s+', ' '))
for c in ATRIBUTOS_CONDICION:
    df_tb = df_tb.withColumn(c, sf.trim(sf.col(c)))
df_tb = df_tb.withColumn('Anio', extraer_anio('Fecha')).drop('Fecha')

filas_antes = df_tb.count()
df_tb = df_tb.dropDuplicates()
filas_despues = df_tb.count()
print(f'Filas de F2: {filas_antes:,} | sin duplicados: {filas_despues:,} | duplicados eliminados: {filas_antes - filas_despues:,}')
print(f'Años en F2: {sorted(r.Anio for r in df_tb.select("Anio").distinct().collect() if r.Anio is not None)}')
print(f'Filas sin año: {df_tb.filter(sf.col("Anio").isNull()).count():,}')

Filas de F2: 449 | sin duplicados: 345 | duplicados eliminados: 104
Años en F2: [2017, 2018, 2019, 2020, 2021]
Filas sin año: 0


### 3.2 Homologación de las condiciones a `Yes` y `No`
**Regla de negocio:** En la fuente de tipos de beneficio siguen existiendo errores de consistencia en los valores de las condiciones, se le pide cambiar los diferentes valores para que queden únicamente "Yes" y "No".

Primero se revisan los valores que trae cada condición. En el entendimiento de datos se encontraron variantes como `Si`, `True`, `Nein` y `False`.

In [7]:
def resumen_valores(df, columnas):
    # Cuenta las filas de cada valor de cada columna, en una sola tabla
    pares = sf.explode(sf.array(*[
        sf.struct(sf.lit(c).alias('Columna'), sf.col(c).cast('string').alias('Valor')) for c in columnas
    ]))
    return (df.select(pares.alias('par'))
              .select('par.Columna', 'par.Valor')
              .groupBy('Columna', 'Valor').count()
              .orderBy('Columna', 'Valor'))

print('Valores de las condiciones antes de homologar:')
resumen_valores(df_tb, CONDICIONES_YES_NO).show(60, truncate=False)

Valores de las condiciones antes de homologar:
+----------------------------------------+-----+-----+
|Columna                                 |Valor|count|
+----------------------------------------+-----+-----+
|EsEHB                                   |No   |225  |
|EsEHB                                   |True |17   |
|EsEHB                                   |Yes  |103  |
|EstaCubiertaPorSeguro                   |False|2    |
|EstaCubiertaPorSeguro                   |No   |30   |
|EstaCubiertaPorSeguro                   |Yes  |313  |
|ExcluidoDelDesembolsoMaximoDentroDeLaRed|No   |327  |
|ExcluidoDelDesembolsoMaximoDentroDeLaRed|Yes  |18   |
|ExcluidoDelDesembolsoMaximoFueraDeLaRed |No   |236  |
|ExcluidoDelDesembolsoMaximoFueraDeLaRed |Yes  |109  |
|TieneLimiteCuantitativo                 |No   |216  |
|TieneLimiteCuantitativo                 |Si   |10   |
|TieneLimiteCuantitativo                 |Yes  |119  |
+----------------------------------------+-----+-----+



Cada valor se pasa a mayúsculas y se busca en una lista de variantes afirmativas y otra de negativas, si aparece un valor que no está en ninguna de las dos, la celda se detiene y lo muestra. Así se puede agregar a la lista que corresponda en vez de cargarlo sin homologar.

In [8]:
VARIANTES_YES = ['YES', 'Y', 'SI', 'SÍ', 'S', 'TRUE', 'VERDADERO', 'JA', '1']
VARIANTES_NO = ['NO', 'N', 'NEIN', 'FALSE', 'FALSO', '0']

def homologar_yes_no(columna):
    valor = sf.upper(sf.col(columna))
    return (sf.when(valor.isin(VARIANTES_YES), 'Yes')
              .when(valor.isin(VARIANTES_NO), 'No')
              .otherwise(sf.col(columna)))

for c in CONDICIONES_YES_NO:
    df_tb = df_tb.withColumn(c, homologar_yes_no(c))

resumen_despues = resumen_valores(df_tb, CONDICIONES_YES_NO)
print('Valores de las condiciones después de homologar:')
resumen_despues.show(60, truncate=False)

sin_homologar = resumen_despues.filter(sf.col('Valor').isNull() | ~sf.col('Valor').isin('Yes', 'No'))
if sin_homologar.count() > 0:
    sin_homologar.show(truncate=False)
    raise ValueError('Hay condiciones con valores distintos de Yes/No: agréguelos a VARIANTES_YES o VARIANTES_NO')
print('Todas las condiciones quedaron con valor Yes o No')

Valores de las condiciones después de homologar:
+----------------------------------------+-----+-----+
|Columna                                 |Valor|count|
+----------------------------------------+-----+-----+
|EsEHB                                   |No   |225  |
|EsEHB                                   |Yes  |120  |
|EstaCubiertaPorSeguro                   |No   |32   |
|EstaCubiertaPorSeguro                   |Yes  |313  |
|ExcluidoDelDesembolsoMaximoDentroDeLaRed|No   |327  |
|ExcluidoDelDesembolsoMaximoDentroDeLaRed|Yes  |18   |
|ExcluidoDelDesembolsoMaximoFueraDeLaRed |No   |236  |
|ExcluidoDelDesembolsoMaximoFueraDeLaRed |Yes  |109  |
|TieneLimiteCuantitativo                 |No   |216  |
|TieneLimiteCuantitativo                 |Yes  |129  |
+----------------------------------------+-----+-----+

Todas las condiciones quedaron con valor Yes o No


### 3.3 Unidad del límite
Cuando el beneficio no tiene límite cuantitativo, `UnidadDelLimite` llega vacía. Se reemplaza por `No aplica` para no cargar vacíos en la dimensión. Además, se unifican las unidades que solo difieren en mayúsculas y minúsculas (por ejemplo `Procedure(s) per lifetime` y `Procedure(s) per Lifetime`), conservando la escritura más frecuente.

También se cuentan los casos incoherentes: con límite pero sin unidad o sin límite pero con unidad. En la ejecución no hubo beneficios con límite y sin unidad, pero sí 16 registros sin límite y con unidad. No se modifican porque no es posible saber si el error está en la condición o en la unidad, y se reportan a RaSA.

In [9]:
unidad_vacia = sf.col('UnidadDelLimite').isNull() | (sf.col('UnidadDelLimite') == '')
df_tb = df_tb.withColumn('UnidadDelLimite', sf.when(unidad_vacia, 'No aplica').otherwise(sf.col('UnidadDelLimite')))

# Unidades que solo difieren en mayúsculas/minúsculas: se conserva la escritura más frecuente
clave_unidad = sf.lower(sf.regexp_replace(sf.col('UnidadDelLimite'), r'\s+', ' '))
variantes_unidad = (df_tb.groupBy('UnidadDelLimite').count()
    .withColumn('clave', clave_unidad)
    .withColumn('orden', sf.row_number().over(
        Window.partitionBy('clave').orderBy(sf.col('count').desc(), sf.col('UnidadDelLimite')))))
unidad_canonica = variantes_unidad.filter(sf.col('orden') == 1).select('clave', sf.col('UnidadDelLimite').alias('UnidadCanonica'))
unidades_unificadas = (variantes_unidad.filter(sf.col('orden') > 1).join(unidad_canonica, 'clave')
                                       .select('UnidadDelLimite', 'UnidadCanonica', 'count'))
print(f'Unidades unificadas por mayúsculas/minúsculas: {unidades_unificadas.count()}')
unidades_unificadas.show(truncate=False)
df_tb = (df_tb.withColumn('clave', clave_unidad)
              .join(unidad_canonica, 'clave', 'left')
              .withColumn('UnidadDelLimite', sf.coalesce('UnidadCanonica', 'UnidadDelLimite'))
              .drop('clave', 'UnidadCanonica'))

df_tb.groupBy('TieneLimiteCuantitativo', 'UnidadDelLimite').count() \
     .orderBy('TieneLimiteCuantitativo', 'UnidadDelLimite').show(50, truncate=False)

con_limite_sin_unidad = df_tb.filter((sf.col('TieneLimiteCuantitativo') == 'Yes') & (sf.col('UnidadDelLimite') == 'No aplica')).count()
sin_limite_con_unidad = df_tb.filter((sf.col('TieneLimiteCuantitativo') == 'No') & (sf.col('UnidadDelLimite') != 'No aplica')).count()
print(f'Con límite cuantitativo pero sin unidad: {con_limite_sin_unidad:,}')
print(f'Sin límite cuantitativo pero con unidad: {sin_limite_con_unidad:,}')

+-----------------------+-------------------------------+-----+
|TieneLimiteCuantitativo|UnidadDelLimite                |count|
+-----------------------+-------------------------------+-----+
|No                     |Dollars per Year               |1    |
|No                     |Exam(s) per Year               |1    |
|No                     |Item(s) per 3 Years            |1    |
|No                     |Item(s) per Year               |1    |
|No                     |Months per 3 Years             |1    |
|No                     |No aplica                      |200  |
|No                     |Procedure(s) per Episode       |1    |
|No                     |Procedure(s) per Year          |1    |
|No                     |Treatment(s) per 6 Months      |1    |
|No                     |Visit(s) per 6 Months          |1    |
|No                     |Visit(s) per Benefit Period    |2    |
|No                     |Visit(s) per Year              |5    |
|Yes                    |Days per Admiss

### 3.4 Una sola combinación de condiciones por tipo de beneficio y año
**Regla de negocio:** En la fuente de tipos de beneficios no deberían existir diferentes condiciones y unidad de límite para el mismo tipo de beneficio en el mismo año. Esto fue arreglado y solo debe existir un valor de condiciones y unidad límite de tipo de beneficio por año.

Esta validación se hace después de homologar, porque antes `Si` y `Yes` contarían como dos valores distintos. Por la misma razón se vuelven a quitar duplicados: dos filas que solo diferían en `Si`/`Yes` ahora son iguales, se espera que no haya ningún par con más de una combinación.

In [10]:
filas_antes = df_tb.count()
df_tb = df_tb.dropDuplicates().cache()
print(f'Filas que quedaron repetidas después de homologar: {filas_antes - df_tb.count():,} | filas finales de F2: {df_tb.count():,}')

conflictos = (df_tb.groupBy('IdTipoBeneficio_T', 'Anio')
                   .agg(sf.countDistinct(*ATRIBUTOS_CONDICION).alias('combinaciones'))
                   .filter(sf.col('combinaciones') > 1))
print(f'Pares (tipo de beneficio, año) con más de una combinación de condiciones: {conflictos.count():,}')
conflictos.orderBy('IdTipoBeneficio_T', 'Anio').show(truncate=False)

Filas que quedaron repetidas después de homologar: 0 | filas finales de F2: 345
Pares (tipo de beneficio, año) con más de una combinación de condiciones: 0
+-----------------+----+-------------+
|IdTipoBeneficio_T|Anio|combinaciones|
+-----------------+----+-------------+
+-----------------+----+-------------+



### 3.5 Dimensión TiposBeneficio
La dimensión guarda solo los atributos que no cambian: la llave natural y el nombre. En el entendimiento de datos el nombre fue 1 a 1 con su ID (178 IDs y 178 nombres), si en la fuente actual un ID tuviera más de un nombre, se conserva el del año más reciente y se reporta.

En la ejecución ningún ID tiene más de un nombre, pero hay 205 IDs y 204 nombres distintos, es decir, dos IDs comparten nombre. Se conservan ambos porque son llaves naturales distintas que F3 puede usar, y se muestran para reportarlos a RaSA.

In [11]:
df_nombres = (df_tb.filter(sf.col('IdTipoBeneficio_T').isNotNull())
                   .select('IdTipoBeneficio_T', 'Nombre', 'Anio').distinct())

ids_con_varios_nombres = (df_nombres.groupBy('IdTipoBeneficio_T')
                                    .agg(sf.countDistinct('Nombre').alias('nombres'))
                                    .filter(sf.col('nombres') > 1))
print(f'IDs con más de un nombre: {ids_con_varios_nombres.count()}')
ids_con_varios_nombres.join(df_nombres, 'IdTipoBeneficio_T').orderBy('IdTipoBeneficio_T', 'Anio').show(truncate=False)

nombre_mas_reciente = Window.partitionBy('IdTipoBeneficio_T').orderBy(sf.col('Anio').desc(), sf.col('Nombre'))
df_tipos_beneficio_final = (df_nombres
    .withColumn('orden', sf.row_number().over(nombre_mas_reciente))
    .filter(sf.col('orden') == 1)
    .withColumn('IdTipoBeneficio_DWH', sf.row_number().over(Window.orderBy('IdTipoBeneficio_T')))
    .select('IdTipoBeneficio_DWH', 'IdTipoBeneficio_T', 'Nombre')
    .cache())

print(f'TiposBeneficio: {df_tipos_beneficio_final.count()} filas | '
      f'IDs distintos: {df_tipos_beneficio_final.select("IdTipoBeneficio_T").distinct().count()} | '
      f'nombres distintos: {df_tipos_beneficio_final.select("Nombre").distinct().count()}')
df_tipos_beneficio_final.show(10, truncate=False)

# Nombres que comparten varios IDs (sin distinguir mayúsculas)
nombres_compartidos = (df_tipos_beneficio_final
    .groupBy(sf.lower('Nombre').alias('NombreNormalizado'))
    .agg(sf.count('*').alias('ids'),
         sf.sort_array(sf.collect_list('IdTipoBeneficio_T')).alias('IdsTipoBeneficio'),
         sf.collect_set('Nombre').alias('Nombres'))
    .filter(sf.col('ids') > 1))
print(f'Nombres compartidos por más de un ID: {nombres_compartidos.count()}')
nombres_compartidos.show(truncate=False)

IDs con más de un nombre: 0
+-----------------+-------+------+----+
|IdTipoBeneficio_T|nombres|Nombre|Anio|
+-----------------+-------+------+----+
+-----------------+-------+------+----+

TiposBeneficio: 205 filas | IDs distintos: 205 | nombres distintos: 204
+-------------------+-----------------+-----------------------------------------------+
|IdTipoBeneficio_DWH|IdTipoBeneficio_T|Nombre                                         |
+-------------------+-----------------+-----------------------------------------------+
|1                  |5                |Abortion For Which Public Funding Is Prohibited|
|2                  |10               |Accidental Dental                              |
|3                  |15               |Acupuncture                                    |
|4                  |20               |Adult Frames And Lenses                        |
|5                  |25               |Allergy Injections                             |
|6                  |30            

### 3.6 Minidimensión MiniCondicionesTipoBeneficio
Se toman las combinaciones distintas de las seis condiciones. La columna `EstaCubiertaPorSeguro` de la fuente se renombra a `EstaCubiertoPorSeguro`, como aparece en el modelo.

In [12]:
df_mini_final = (df_tb.select(*ATRIBUTOS_CONDICION).distinct()
    .withColumn('IdCondicionesBeneficios_DWH', sf.row_number().over(Window.orderBy(*ATRIBUTOS_CONDICION)))
    .select('IdCondicionesBeneficios_DWH',
            sf.col('EstaCubiertaPorSeguro').alias('EstaCubiertoPorSeguro'),
            'EsEHB',
            'TieneLimiteCuantitativo',
            'ExcluidoDelDesembolsoMaximoDentroDeLaRed',
            'ExcluidoDelDesembolsoMaximoFueraDeLaRed',
            'UnidadDelLimite')
    .cache())

print(f'MiniCondicionesTipoBeneficio: {df_mini_final.count()} combinaciones distintas de condiciones')
df_mini_final.show(50, truncate=False)

MiniCondicionesTipoBeneficio: 96 combinaciones distintas de condiciones
+---------------------------+---------------------+-----+-----------------------+----------------------------------------+---------------------------------------+-------------------------------+
|IdCondicionesBeneficios_DWH|EstaCubiertoPorSeguro|EsEHB|TieneLimiteCuantitativo|ExcluidoDelDesembolsoMaximoDentroDeLaRed|ExcluidoDelDesembolsoMaximoFueraDeLaRed|UnidadDelLimite                |
+---------------------------+---------------------+-----+-----------------------+----------------------------------------+---------------------------------------+-------------------------------+
|1                          |No                   |No   |No                     |No                                      |No                                     |No aplica                      |
|2                          |No                   |No   |No                     |No                                      |Yes                       

### 3.7 Mapa de condiciones para las tablas de hechos
`HechoPlanesTiposBeneficio` y `HechoHistCondicionesTiposBeneficio` necesitan saber qué combinación de condiciones tenía cada tipo de beneficio en cada año, por lo que el mapa se arma cruzando F2 ya homologada con la minidimensión por las seis condiciones, también incluye `IdFecha` con el corte anual que coincide con las fechas de F3 y con `Fecha_ELT`.

El mapa no se carga a la base porque no es una tabla del modelo. Quien cargue los hechos puede reconstruirlo leyendo `MiniCondicionesTipoBeneficio_ELT` y aplicando a F2 las transformaciones 3.1 a 3.4. Se valida que:

1. cada par (tipo de beneficio, año) tenga una sola condición, para que no se dupliquen filas en los hechos.
2. todos los pares (tipo de beneficio, año) que usa F3 encuentren su condición en F2.

En la ejecución, 265 de los 610 pares de F3 no tienen condición en F2 para ese año, porque F2 no trae todos los beneficios en todos los años (por ejemplo, el beneficio 5 aparece en 2017, 2018 y 2020, pero F3 también lo usa en 2019 y 2021). Con el manejo de historia tipo 4 una condición sigue vigente hasta que cambia, por eso se recomienda que la carga de los hechos use la condición del último año anterior disponible. La celda cuenta cuántos pares se recuperan así y muestra los que no tienen ninguna condición anterior, para los que se sugiere un registro desconocido.

In [13]:
df_mapa_condiciones = (df_tb
    .join(df_mini_final.withColumnRenamed('EstaCubiertoPorSeguro', 'EstaCubiertaPorSeguro'),
          on=ATRIBUTOS_CONDICION, how='inner')
    .select('IdTipoBeneficio_T',
            'Anio',
            (sf.col('Anio') * 10000 + 1231).alias('IdFecha'),
            sf.col('IdCondicionesBeneficios_DWH').alias('IdCondicionesTipoBeneficio_DWH'))
    .distinct()
    .cache())
df_mapa_condiciones.orderBy('IdTipoBeneficio_T', 'Anio').show(10)

# Validación 1: una sola condición por par (tipo de beneficio, año)
pares_ambiguos = (df_mapa_condiciones.groupBy('IdTipoBeneficio_T', 'Anio').count()
                                     .filter(sf.col('count') > 1).count())
print(f'Pares (tipo de beneficio, año) con más de una condición: {pares_ambiguos:,}')

# Validación 2: los pares de F3 encuentran su condición
df_pares_f3 = df_planes_raw.select('IdTipoBeneficio_T', extraer_anio('Fecha').alias('Anio')).distinct()
pares_sin_condicion = df_pares_f3.join(df_mapa_condiciones, ['IdTipoBeneficio_T', 'Anio'], 'left_anti')
print(f'Pares (tipo de beneficio, año) en F3: {df_pares_f3.count():,} | sin condición en F2: {pares_sin_condicion.count():,}')
pares_sin_condicion.orderBy('IdTipoBeneficio_T', 'Anio').show(truncate=False)

# Pares sin condición: ¿hay una condición de un año anterior que siga vigente?
anios_con_condicion = df_mapa_condiciones.select('IdTipoBeneficio_T', sf.col('Anio').alias('AnioCondicion'))
vigencia = (pares_sin_condicion.join(anios_con_condicion, 'IdTipoBeneficio_T', 'left')
    .groupBy('IdTipoBeneficio_T', 'Anio')
    .agg(sf.max(sf.when(sf.col('AnioCondicion') < sf.col('Anio'), sf.col('AnioCondicion'))).alias('AnioVigente')))
print(f'  se recuperan con la condición vigente de un año anterior: {vigencia.filter(sf.col("AnioVigente").isNotNull()).count():,}')
print(f'  sin ninguna condición anterior: {vigencia.filter(sf.col("AnioVigente").isNull()).count():,}')
vigencia.filter(sf.col('AnioVigente').isNull()).orderBy('IdTipoBeneficio_T', 'Anio').show(truncate=False)

+-----------------+----+--------+------------------------------+
|IdTipoBeneficio_T|Anio| IdFecha|IdCondicionesTipoBeneficio_DWH|
+-----------------+----+--------+------------------------------+
|                5|2017|20171231|                             1|
|                5|2018|20181231|                             3|
|                5|2020|20201231|                            10|
|               10|2017|20171231|                            61|
|               10|2018|20181231|                            12|
|               10|2019|20191231|                            42|
|               10|2021|20211231|                            63|
|               15|2017|20171231|                             1|
|               15|2018|20181231|                             3|
|               15|2019|20191231|                            24|
+-----------------+----+--------+------------------------------+
only showing top 10 rows
Pares (tipo de beneficio, año) con más de una condición: 0
Pares 

### 3.8 Dimensión Proveedor
Por simplificación la dimensión solo tiene identificadores y estos se obtienen de F3. En el entendimiento de datos, `IdProveedor_T` coincidió en el 100 % de las filas con los 5 primeros caracteres de `IdPlan_T`, por eso ese prefijo se usa para completar un proveedor faltante, también se reporta cuántas filas no coinciden.

In [14]:
# El prefijo del plan se convierte al mismo tipo de dato que IdProveedor_T para poder compararlos
tipo_proveedor = df_planes_raw.schema['IdProveedor_T'].dataType
df_prov = (df_planes_raw.select('IdProveedor_T', 'IdPlan_T').dropDuplicates()
           .withColumn('ProveedorDesdePlan', sf.substring(sf.trim(sf.col('IdPlan_T')), 1, 5).cast(tipo_proveedor)))

sin_proveedor = df_prov.filter(sf.col('IdProveedor_T').isNull()).count()
sin_proveedor_ni_plan = df_prov.filter(sf.col('IdProveedor_T').isNull() & sf.col('ProveedorDesdePlan').isNull()).count()
no_coinciden = df_prov.filter(sf.col('IdProveedor_T').isNotNull() & sf.col('ProveedorDesdePlan').isNotNull()
                              & (sf.col('IdProveedor_T') != sf.col('ProveedorDesdePlan'))).count()
print(f'Pares (proveedor, plan) distintos en F3: {df_prov.count():,}')
print(f'  sin IdProveedor_T (se completa con el prefijo de IdPlan_T): {sin_proveedor:,}')
print(f'  sin IdProveedor_T ni IdPlan_T (no se pueden recuperar): {sin_proveedor_ni_plan:,}')
print(f'  con IdProveedor_T distinto del prefijo del plan: {no_coinciden:,}')

df_proveedor_final = (df_prov
    .select(sf.coalesce('IdProveedor_T', 'ProveedorDesdePlan').alias('IdProveedor_T'))
    .filter(sf.col('IdProveedor_T').isNotNull())
    .distinct()
    .withColumn('IdProveedor_DWH', sf.row_number().over(Window.orderBy('IdProveedor_T')))
    .select('IdProveedor_DWH', 'IdProveedor_T')
    .cache())

print(f'Proveedor: {df_proveedor_final.count()} proveedores')
df_proveedor_final.show(10)

Pares (proveedor, plan) distintos en F3: 654
  sin IdProveedor_T (se completa con el prefijo de IdPlan_T): 0
  sin IdProveedor_T ni IdPlan_T (no se pueden recuperar): 0
  con IdProveedor_T distinto del prefijo del plan: 0
Proveedor: 171 proveedores
+---------------+-------------+
|IdProveedor_DWH|IdProveedor_T|
+---------------+-------------+
|              1|        10207|
|              2|        11269|
|              3|        11469|
|              4|        11512|
|              5|        12303|
|              6|        12379|
|              7|        12858|
|              8|        14002|
|              9|        14609|
|             10|        15560|
+---------------+-------------+
only showing top 10 rows


## 4. Carga
La carga es en modo `overwrite` con `truncate`: cada ejecución vacía las tres tablas y las vuelve a llenar, así el notebook se puede correr de nuevo sin duplicar filas.

In [15]:
tablas_destino = {
    'TiposBeneficio_ELT': df_tipos_beneficio_final,
    'MiniCondicionesTipoBeneficio_ELT': df_mini_final,
    'Proveedor_ELT': df_proveedor_final,
}

for tabla, df in tablas_destino.items():
    guardar_db(dest_db_connection_string, df, f'{db_destino}.{tabla}', db_user, db_psswd, modo='overwrite')
    print(f'{tabla}: {df.count():,} filas cargadas')

print('¡Proceso ETL de TiposBeneficio, MiniCondicionesTipoBeneficio y Proveedor finalizado!')

TiposBeneficio_ELT: 205 filas cargadas
MiniCondicionesTipoBeneficio_ELT: 96 filas cargadas
Proveedor_ELT: 171 filas cargadas
¡Proceso ETL de TiposBeneficio, MiniCondicionesTipoBeneficio y Proveedor finalizado!


### Verificación de la carga
Se cuentan las filas de cada tabla en la base y se comparan con las del DataFrame cargado. Ambos conteos deben coincidir.

In [16]:
for tabla, df in tablas_destino.items():
    sql_conteo = f'(SELECT COUNT(*) AS filas FROM {db_destino}.{tabla}) AS conteo'
    filas_bd = obtener_dataframe_de_bd(dest_db_connection_string, sql_conteo, db_user, db_psswd).first()['filas']
    filas_df = df.count()
    estado = 'OK' if filas_bd == filas_df else 'REVISAR'
    print(f'{tabla}: DataFrame {filas_df:,} | base de datos {filas_bd:,} | {estado}')

TiposBeneficio_ELT: DataFrame 205 | base de datos 205 | OK
MiniCondicionesTipoBeneficio_ELT: DataFrame 96 | base de datos 96 | OK
Proveedor_ELT: DataFrame 171 | base de datos 171 | OK


# Cierre de PySpark

In [17]:
spark.stop()